# 03 · Explanations

Computes and compares SHAP/gradient attributions across the three modelling
approaches. Tests the explainability hypotheses from the project plan.

| Section | Hypothesis tested |
|---|---|
| 2. Attributions | — (computation) |
| 3. Global A vs B vs C | EH3 — native alignment |
| 4. Cross-method consistency | EH0 prerequisite |
| 5. Cross-seed stability | EH1 — native stability |
| 6. Target disentanglement | EH2 — feature specificity |
| 7. Attribution entropy | EH4 — locality |
| 8. Fragment visualisation | — (qualitative) |

**Input:** `data.pkl`, `models.pkl`

## 0. Imports

In [ ]:
import sys, pickle, random, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr, pearsonr, kendalltau
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import r2_score
from collections import defaultdict
import itertools
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from utils import build_feature_matrix, expand_shap_to_fp, SEED, N_FP

from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.Draw import rdMolDraw2D

import torch
import torch.nn as nn

import shap

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

seed_everything()
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
_gen   = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=N_FP)
print(f"Device: {DEVICE}  |  SEED: {SEED}")

## 1. Load data and models

In [ ]:
# Model architecture definitions must exist before pickle.load
# so PyTorch can reconstruct the saved model objects.

import torch.nn as nn

class PyramidMLP(nn.Module):
    """pyramid_down — 512→256→128, GELU, BatchNorm, single output."""
    def __init__(self, input_dim, dropout=0.25, n_outputs=1):
        super().__init__()
        dims = [512, 256, 128]
        layers, in_d = [], input_dim
        for h in dims:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h),
                       nn.GELU(), nn.Dropout(dropout)]
            in_d = h
        self.encoder = nn.Sequential(*layers)
        self.head     = nn.Linear(in_d, 1)
    def forward(self, x):
        return self.head(self.encoder(x)).squeeze(-1)


class TwoHeadMLP(nn.Module):
    """Shared encoder (512→256), two separate heads (256→128→1) — Approach C."""
    def __init__(self, input_dim, dropout=0.25):
        super().__init__()
        shared = [512, 256]
        layers, in_d = [], input_dim
        for h in shared:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h),
                       nn.GELU(), nn.Dropout(dropout)]
            in_d = h
        self.encoder  = nn.Sequential(*layers)
        self.head_d2  = nn.Sequential(
            nn.Linear(in_d, 128), nn.GELU(), nn.Linear(128, 1))
        self.head_sht = nn.Sequential(
            nn.Linear(in_d, 128), nn.GELU(), nn.Linear(128, 1))
    def forward(self, x):
        h = self.encoder(x)
        return self.head_d2(h).squeeze(-1), self.head_sht(h).squeeze(-1)


# Generic flexible MLP used by the overnight search
class FlexMLP(nn.Module):
    """Variable architecture from HP search — reconstructed at load time."""
    def __init__(self, input_dim, dims, dropout, norm='batch',
                 activation='gelu', use_skip=False):
        super().__init__()
        act = {'gelu': nn.GELU(), 'silu': nn.SiLU(), 'mish': nn.Mish()}[activation]
        self.use_skip = use_skip
        self.blocks   = nn.ModuleList()
        in_d = input_dim
        for h in dims:
            nl = nn.BatchNorm1d(h) if norm == 'batch' else nn.LayerNorm(h)
            self.blocks.append(nn.Sequential(
                nn.Linear(in_d, h), nl, act, nn.Dropout(dropout)))
            in_d = h
        self.head      = nn.Linear(in_d, 1)
        self.skip_proj = (nn.Linear(input_dim, dims[-1], bias=False)
                          if use_skip and input_dim != dims[-1] else None)
    def forward(self, x):
        h = x
        for i, block in enumerate(self.blocks):
            h_new = block(h)
            if self.use_skip and i == len(self.blocks)-1 and self.skip_proj:
                h_new = h_new + self.skip_proj(x)
            h = h_new
        return self.head(h).squeeze(-1)

print("Model classes defined.")

In [ ]:
with open('data.pkl',   'rb') as f: data   = pickle.load(f)
with open('models.pkl', 'rb') as f: models = pickle.load(f)

merged    = data['merged']
sp        = data['splits']
sel       = models['selectors']
sc        = models['scalers']

smiles_ov  = merged['curated_smiles'].tolist()
smiles_d2  = data['d2']['curated_smiles'].tolist()
smiles_sht = data['sht']['curated_smiles'].tolist()

X_ov  = data['X_ov']
X_d2  = data['X_d2']
X_sht = data['X_sht']

tr_ov, va_ov, te_ov = sp['tr_ov'], sp['va_ov'], sp['te_ov']
tr_d2, va_d2, te_d2 = sp['tr_d2'], sp['va_d2'], sp['te_d2']
tr_st, va_st, te_st = sp['tr_sht'], sp['va_sht'], sp['te_sht']

# Feature matrices
X_feat_ov  = build_feature_matrix(X_ov,  smiles_ov,  sel['overlap'], sc['overlap'])
X_feat_d2  = build_feature_matrix(X_d2,  smiles_d2,  sel['D2'],      sc['D2'])
X_feat_sht = build_feature_matrix(X_sht, smiles_sht, sel['5HT2A'],   sc['5HT2A'])

# Test splits only — attributions never touch train or val
X_te_ov  = X_feat_ov[te_ov]
X_te_d2  = X_feat_d2[te_d2]
X_te_sht = X_feat_sht[te_st]

smiles_te_ov  = [smiles_ov[i]  for i in te_ov]
smiles_te_d2  = [smiles_d2[i]  for i in te_d2]
smiles_te_sht = [smiles_sht[i] for i in te_st]

# Overlap test compounds — same rows for A, B, C comparisons
smiles_te_ov = [smiles_ov[i] for i in te_ov]

def features_for_smiles(smiles_list, smiles_ref, X_ref):
    """Map overlap smiles to receptor-specific feature rows (aligned order)."""
    idx = {s: i for i, s in enumerate(smiles_ref)}
    return X_ref[[idx[s] for s in smiles_list]]

X_te_d2_ov  = features_for_smiles(smiles_te_ov, smiles_d2,  X_feat_d2)
X_te_sht_ov = features_for_smiles(smiles_te_ov, smiles_sht, X_feat_sht)
assert len(X_te_d2_ov) == len(X_te_ov) == len(X_te_sht_ov)

# Background for interventional SHAP — 200 training compounds
rng    = np.random.RandomState(SEED)
bg_ov  = X_feat_ov[tr_ov][rng.choice(len(tr_ov), 200, replace=False)]
bg_d2  = X_feat_d2[tr_d2][rng.choice(len(tr_d2), 200, replace=False)]
bg_sht = X_feat_sht[tr_st][rng.choice(len(tr_st), 200, replace=False)]

# ── Load best models ──────────────────────────────────────────────────────────
# tree_best is a dict keyed by task label, matching tree_tasks in notebook 02
tb = models['tree_best']
tree_A    = tb['A — direct Δ']
tree_B_d2  = tb['B — D2']
tree_B_sht = tb['B — 5HT2A']
tree_C_d2  = tb['C — D2']
tree_C_sht = tb['C — 5HT2A']

# MLP models (PyramidMLP / TwoHeadMLP — class defs required above this cell)
mlp_A     = models['mlp_A']
mlp_B_d2  = models['mlp_B_d2']
mlp_B_sht = models['mlp_B_sht']
mlp_C     = models['mlp_C']

for m in [mlp_A, mlp_B_d2, mlp_B_sht, mlp_C]:
    m.to(DEVICE).eval()

to_t = lambda a: torch.tensor(a, dtype=torch.float32).to(DEVICE)

# Print available keys for reference
print(f"tree_best tasks: {list(tb.keys())}")
print(f"MLP models: mlp_A, mlp_B_d2, mlp_B_sht, mlp_C")
print(f"Test set:  overlap={len(X_te_ov):,}  D2={len(X_te_d2):,}  5HT2A={len(X_te_sht):,}")
print(f"Background: {len(bg_ov)} compounds per model")
print("Models loaded and ready.")

## 2. Compute attributions

Tree models: **interventional SHAP** with 200-compound background.
Eliminates ghost-bit artefacts from tree_path_dependent mode.

MLP models: **GradientSHAP** (captum) — gradient-based Shapley approximation,
same 200-compound background. Returns attribution in the same 2048-bit space.

In [ ]:
# ── 2a. Tree SHAP — interventional ────────────────────────────────────────────
print("Computing tree SHAP (interventional)...")

kw = dict(feature_perturbation='interventional')

sv_tree_A   = shap.TreeExplainer(tree_A,   data=bg_ov,  **kw).shap_values(X_te_ov)
shap_tree_A = expand_shap_to_fp(sv_tree_A, sel['overlap'])

sv_tree_Bd2  = shap.TreeExplainer(tree_B_d2,  data=bg_d2,  **kw).shap_values(X_te_d2_ov)
sv_tree_Bsht = shap.TreeExplainer(tree_B_sht, data=bg_sht, **kw).shap_values(X_te_sht_ov)
shap_tree_B  = (expand_shap_to_fp(sv_tree_Bsht, sel['5HT2A']) -
                expand_shap_to_fp(sv_tree_Bd2,  sel['D2']))

print(f"  Tree SHAP shapes: A={shap_tree_A.shape}  B={shap_tree_B.shape}")

In [ ]:
# ── 2b. GradientSHAP for MLP ──────────────────────────────────────────────────
# Patch numpy 2.x compatibility
import numpy as np
if not hasattr(np, 'bool'):   np.bool   = bool
if not hasattr(np, 'int'):    np.int    = int
if not hasattr(np, 'float'):  np.float  = float
if not hasattr(np, 'object'): np.object = object

from captum.attr import GradientShap

def mlp_gradshap(model, X, background, n_samples=50, seed=SEED):
    """Compute GradientSHAP attributions. Returns (n, n_features) array."""
    seed_everything(seed)
    X_t  = to_t(X)
    bg_t = to_t(background)

    gs   = GradientShap(model)
    attr = gs.attribute(X_t, bg_t,
                        n_samples=n_samples,
                        stdevs=0.09,
                        return_convergence_delta=False)
    return attr.detach().cpu().numpy()

def mlp_gradshap_head(model, X, background, head='d2', n_samples=50, seed=SEED):
    """GradientSHAP for one head of the two-headed MLP (Approach C)."""
    seed_everything(seed)
    X_t  = to_t(X)
    bg_t = to_t(background)

    idx = 0 if head == 'd2' else 1
    wrapper = lambda x: model(x)[idx].unsqueeze(-1)

    class HeadWrapper(nn.Module):
        def __init__(self, m, i): super().__init__(); self.m=m; self.i=i
        def forward(self, x): return self.m(x)[self.i].unsqueeze(-1)

    hw = HeadWrapper(model, idx)
    gs = GradientShap(hw)
    attr = gs.attribute(X_t, bg_t,
                        n_samples=n_samples,
                        stdevs=0.09,
                        return_convergence_delta=False)
    return attr.detach().cpu().numpy()

print("Computing MLP GradientSHAP...")
print("  MLP A...")
gs_mlp_A  = mlp_gradshap(mlp_A, X_te_ov, bg_ov)
shap_mlp_A = expand_shap_to_fp(gs_mlp_A, sel['overlap'])

print("  MLP B (D2 and 5HT2A)...")
gs_mlp_Bd2  = mlp_gradshap(mlp_B_d2,  X_te_d2_ov,  bg_d2)
gs_mlp_Bsht = mlp_gradshap(mlp_B_sht, X_te_sht_ov, bg_sht)
shap_mlp_B  = (expand_shap_to_fp(gs_mlp_Bsht, sel['5HT2A']) -
               expand_shap_to_fp(gs_mlp_Bd2,  sel['D2']))

print("  MLP C (two-headed)...")
gs_mlp_Cd2  = mlp_gradshap_head(mlp_C, X_te_ov, bg_ov, head='d2')
gs_mlp_Csht = mlp_gradshap_head(mlp_C, X_te_ov, bg_ov, head='sht')
shap_mlp_C  = (expand_shap_to_fp(gs_mlp_Csht, sel['overlap']) -
               expand_shap_to_fp(gs_mlp_Cd2,  sel['overlap']))

print(f"  MLP GradSHAP shapes: A={shap_mlp_A.shape}  B={shap_mlp_B.shape}  C={shap_mlp_C.shape}")

## 3. Global A vs B vs C comparison

Spearman ρ, Pearson r, and mean cosine similarity between global importance
vectors and per-compound attribution vectors.

**Global importance** = mean |SHAP| per feature across test compounds.
**Compound-level** = cosine similarity between the two full attribution
vectors for the same compound.

In [ ]:
def compare_approaches(sA, sB, label_a='A', label_b='B', n_boot=500):
    """
    Compare two SHAP matrices (n_compounds, 2048).
    Returns dict of Spearman rho, Pearson r, mean cosine (compound-level),
    and bootstrap 95% CIs for each.
    """
    imp_A = np.abs(sA).mean(axis=0)
    imp_B = np.abs(sB).mean(axis=0)

    rho_global, _  = spearmanr(imp_A, imp_B)
    r_global,   _  = pearsonr(imp_A, imp_B)
    cos_per_cpd    = cosine_similarity(sA, sB).diagonal()
    cos_mean       = cos_per_cpd.mean()

    # Bootstrap CIs
    rng_b = np.random.RandomState(SEED)
    n     = len(sA)
    boot_rho, boot_cos = [], []
    for _ in range(n_boot):
        idx  = rng_b.choice(n, n, replace=True)
        iA   = np.abs(sA[idx]).mean(axis=0)
        iB   = np.abs(sB[idx]).mean(axis=0)
        boot_rho.append(spearmanr(iA, iB)[0])
        boot_cos.append(cosine_similarity(sA[idx], sB[idx]).diagonal().mean())

    return {
        'rho':     rho_global,
        'rho_ci':  (np.percentile(boot_rho, 2.5), np.percentile(boot_rho, 97.5)),
        'r':       r_global,
        'cos':     cos_mean,
        'cos_ci':  (np.percentile(boot_cos, 2.5), np.percentile(boot_cos, 97.5)),
        'cos_dist': cos_per_cpd,
    }

print("Global comparison (test set, interventional SHAP / GradientSHAP)")
print("="*70)
print(f"{'Pair':25s} | {'Spearman ρ':>22s} | {'Mean cosine':>22s}")
print("-"*70)

comparisons = {}
for (lA, sA), (lB, sB) in [
    (('Tree A', shap_tree_A), ('Tree B', shap_tree_B)),
    (('MLP A',  shap_mlp_A),  ('MLP B',  shap_mlp_B)),
    (('MLP A',  shap_mlp_A),  ('MLP C',  shap_mlp_C)),
    (('MLP B',  shap_mlp_B),  ('MLP C',  shap_mlp_C)),
]:
    key = f"{lA} vs {lB}"
    res = compare_approaches(sA, sB)
    comparisons[key] = res
    rlo, rhi = res['rho_ci']
    clo, chi = res['cos_ci']
    print(f"  {key:23s} | {res['rho']:6.4f} [{rlo:.4f},{rhi:.4f}] "
          f"| {res['cos']:6.4f} [{clo:.4f},{chi:.4f}]")

In [ ]:
# Cosine distribution plots per pair
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
pairs = list(comparisons.items())
colors = ['#1D9E75', '#534AB7', '#D85A30', '#D97706']

for ax, (key, res), col in zip(axes, pairs, colors):
    ax.hist(res['cos_dist'], bins=40, color=col, alpha=0.8, edgecolor='none')
    ax.axvline(res['cos'], color='black', lw=2, label=f"mean={res['cos']:.3f}")
    ax.axvline(0, color='gray', lw=1, linestyle='--')
    ax.set(xlabel='Cosine similarity', ylabel='Count',
           title=key, xlim=[-1, 1])
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Per-compound cosine similarity distributions\n'
             '(each dot = one test compound, two attribution vectors compared)',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_03_cosine_distributions.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. Cross-method consistency (EH0 prerequisite)

Computes the grand matrix of Spearman ρ between global importance vectors
across all explanation methods for Approach A.

High agreement (ρ > 0.9) means the signal is a property of the data and model,
not an artefact of the explanation method chosen.

In [ ]:
from captum.attr import IntegratedGradients, NoiseTunnel, DeepLift

def ig_attributions(model, X, baselines=None, n_steps=50):
    ig   = IntegratedGradients(model)
    base = baselines if baselines is not None else torch.zeros_like(to_t(X))
    attr = ig.attribute(to_t(X), baselines=base,
                        n_steps=n_steps, return_convergence_delta=False)
    return attr.detach().cpu().numpy()

def deeplift_attributions(model, X, baselines=None):
    dl   = DeepLift(model)
    base = baselines if baselines is not None else torch.zeros_like(to_t(X))
    attr = dl.attribute(to_t(X), baselines=base)
    return attr.detach().cpu().numpy()

# Use a subsample for speed
N_CROSS = min(300, len(X_te_ov))
idx_sub  = np.random.RandomState(SEED).choice(len(X_te_ov), N_CROSS, replace=False)
X_sub    = X_te_ov[idx_sub]
base_sub = torch.zeros(1, X_sub.shape[1]).to(DEVICE)

print(f"Computing cross-method attributions on {N_CROSS} test compounds...")

method_attrs = {
    'GradSHAP (MLP A)': shap_mlp_A[idx_sub],
}

print("  Integrated Gradients...")
ig_raw  = ig_attributions(mlp_A, X_sub, baselines=base_sub)
method_attrs['IntGrad (MLP A)'] = expand_shap_to_fp(ig_raw, sel['overlap'])

print("  DeepLIFT...")
try:
    dl_raw  = deeplift_attributions(mlp_A, X_sub, baselines=base_sub)
    method_attrs['DeepLIFT (MLP A)'] = expand_shap_to_fp(dl_raw, sel['overlap'])
except Exception as e:
    print(f"    DeepLIFT failed: {e} — skipping")

print("  Tree SHAP (Approach A)...")
method_attrs['TreeSHAP (Tree A)'] = shap_tree_A[idx_sub]

# Grand matrix
methods = list(method_attrs.keys())
n_m     = len(methods)
mat_rho = np.zeros((n_m, n_m))

for i, j in itertools.combinations(range(n_m), 2):
    iA = np.abs(method_attrs[methods[i]]).mean(axis=0)
    iB = np.abs(method_attrs[methods[j]]).mean(axis=0)
    rho, _ = spearmanr(iA, iB)
    mat_rho[i, j] = mat_rho[j, i] = rho
np.fill_diagonal(mat_rho, 1.0)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(mat_rho, vmin=0.5, vmax=1.0, cmap='YlGn', aspect='auto')
ax.set_xticks(range(n_m)); ax.set_xticklabels(methods, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(n_m)); ax.set_yticklabels(methods, fontsize=9)
for i in range(n_m):
    for j in range(n_m):
        ax.text(j, i, f"{mat_rho[i,j]:.3f}", ha='center', va='center', fontsize=10,
                color='white' if mat_rho[i,j] > 0.8 else 'black')
plt.colorbar(im, ax=ax, label='Spearman ρ')
ax.set_title('Cross-method global importance agreement (Approach A)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_03_cross_method.png', dpi=130, bbox_inches='tight')
plt.show()

print(f"\nMin off-diagonal ρ: {mat_rho[mat_rho < 1].min():.4f}")
print(f"Mean off-diagonal ρ: {mat_rho[mat_rho < 1].mean():.4f}")

## 5. EH1 — Cross-seed stability

**Hypothesis:** Explanations of selectivity are most stable when selectivity
is learned directly, and least stable for independent models.

$$\mathbb{E}_{i,j}[\cos(\phi_i, \phi_j)]_{direct} > \mathbb{E}_{i,j}[\cos(\phi_i, \phi_j)]_{multi} > \mathbb{E}_{i,j}[\cos(\phi_i, \phi_j)]_{post}$$

Measured via Kendall τ between top-20 global importance rankings across 5 seeds.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

N_SEEDS   = 5
CV_SEEDS  = [42, 7, 13, 99, 123]
TOP_K     = 20

y_del = merged['delta'].values
y_d2  = data['d2']['pChEMBL'].values
y_sht = data['sht']['pChEMBL'].values
y_d2_ov  = merged['pChEMBL_D2'].values
y_sht_ov = merged['pChEMBL_5HT2A'].values

# EH1 refits RF across seeds — only copy params if best tree A is actually RF
_rf_defaults = dict(n_estimators=300, max_depth=10, min_samples_leaf=5,
                    max_features=0.2, n_jobs=-1)
_rf_valid = set(RandomForestRegressor().get_params().keys())
best_tree_A = models.get('best_tree_A')
if isinstance(best_tree_A, RandomForestRegressor):
    rf_params = {k: v for k, v in best_tree_A.get_params().items()
                 if k in _rf_valid and k not in ('random_state', 'n_jobs')}
else:
    rf_params = _rf_defaults.copy()
    if best_tree_A is not None:
        print(f"  Note: best_tree_A is {type(best_tree_A).__name__}; "
              f"EH1 uses RF defaults for cross-seed comparison")

def top_k_ranking(shap_mat, k=TOP_K):
    imp = np.abs(shap_mat).mean(axis=0)
    return np.argsort(imp)[::-1][:k].tolist()

def mean_kendall_tau(rankings):
    """Mean Kendall τ over all pairs of rankings."""
    taus = []
    for r1, r2 in itertools.combinations(rankings, 2):
        union = list(set(r1) | set(r2))
        v1 = [r1.index(b)+1 if b in r1 else TOP_K+1 for b in union]
        v2 = [r2.index(b)+1 if b in r2 else TOP_K+1 for b in union]
        tau, _ = kendalltau(v1, v2)
        taus.append(tau)
    return np.mean(taus), np.std(taus)

print(f"Cross-seed stability: {N_SEEDS} seeds, top-{TOP_K} ranking, Kendall τ")
print("="*55)

stability_results = {}
for approach, fit_fn in [
    ('A — direct',   lambda seed: _fit_and_shap_A(seed)),
    ('B — post-hoc', lambda seed: _fit_and_shap_B(seed)),
    ('C — multi',    lambda seed: _fit_and_shap_C(seed)),
]:
    # Define fit functions inline
    def _fit_and_shap_A(seed):
        m = RandomForestRegressor(**rf_params, random_state=seed)
        m.fit(X_feat_ov[tr_ov], y_del[tr_ov])
        sv = shap.TreeExplainer(m, data=bg_ov,
                                 feature_perturbation='interventional').shap_values(X_te_ov)
        return expand_shap_to_fp(sv, sel['overlap'])

    def _fit_and_shap_B(seed):
        md2  = RandomForestRegressor(**rf_params, random_state=seed)
        msht = RandomForestRegressor(**rf_params, random_state=seed)
        md2.fit(X_feat_d2[tr_d2],   y_d2[tr_d2])
        msht.fit(X_feat_sht[tr_st], y_sht[tr_st])
        sv_d2  = shap.TreeExplainer(md2,  data=bg_d2,  feature_perturbation='interventional').shap_values(X_te_d2_ov)
        sv_sht = shap.TreeExplainer(msht, data=bg_sht, feature_perturbation='interventional').shap_values(X_te_sht_ov)
        return (expand_shap_to_fp(sv_sht, sel['5HT2A']) -
                expand_shap_to_fp(sv_d2,  sel['D2']))

    def _fit_and_shap_C(seed):
        Y = np.column_stack([y_d2_ov[tr_ov], y_sht_ov[tr_ov]])
        m = RandomForestRegressor(**rf_params, random_state=seed)
        m.fit(X_feat_ov[tr_ov], Y)
        sv = shap.TreeExplainer(m, data=bg_ov,
                                 feature_perturbation='interventional').shap_values(X_te_ov)
        if isinstance(sv, list):
            return (expand_shap_to_fp(sv[1], sel['overlap']) -
                    expand_shap_to_fp(sv[0], sel['overlap']))
        return expand_shap_to_fp(sv[:,:,1] - sv[:,:,0], sel['overlap'])

    fn = {'A — direct': _fit_and_shap_A,
          'B — post-hoc': _fit_and_shap_B,
          'C — multi': _fit_and_shap_C}[approach]

    print(f"  {approach}...", end='', flush=True)
    rankings = []
    for seed in CV_SEEDS:
        shap_mat = fn(seed)
        rankings.append(top_k_ranking(shap_mat))
        print('.', end='', flush=True)
    tau_mean, tau_std = mean_kendall_tau(rankings)
    stability_results[approach] = (tau_mean, tau_std, rankings)
    print(f"  τ = {tau_mean:.4f} ± {tau_std:.4f}")

print("\nEH1 result:")
for approach, (tau, std, _) in stability_results.items():
    verdict = '↑ stable' if tau > 0.6 else '~ moderate' if tau > 0.4 else '↓ unstable'
    print(f"  {approach:18s}: τ = {tau:.4f} ± {std:.4f}  {verdict}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
names  = list(stability_results.keys())
taus   = [stability_results[n][0] for n in names]
stds   = [stability_results[n][1] for n in names]
colors = ['#1D9E75', '#D85A30', '#534AB7']
ax.bar(names, taus, yerr=stds, color=colors, alpha=0.85,
       edgecolor='black', capsize=5)
ax.axhline(0.6, color='black', lw=1, linestyle=':', label='τ=0.6 stable threshold')
ax.set(ylabel='Mean Kendall τ', ylim=[0, 1.05],
       title='EH1 — Cross-seed explanation stability')
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('fig_03_eh1_stability.png', dpi=130, bbox_inches='tight')
plt.show()

## 6. EH2 — Target disentanglement

**Hypothesis:** Multi-task learning produces more target-specific explanations
than independent models.

$$\cos(\phi^A, \phi^B)_{multi} < \cos(\phi^A, \phi^B)_{post}$$

Measured as cosine similarity between the D2 attribution vector and the 5HT₂A
attribution vector for the same compound, averaged over the test set.

In [ ]:
# For Approach B: φ_D2 = SHAP from D2 model, φ_5HT2A = SHAP from 5HT2A model
# Both expanded to 2048-bit space
shap_B_d2_full  = expand_shap_to_fp(sv_tree_Bd2,  sel['D2'])
shap_B_sht_full = expand_shap_to_fp(sv_tree_Bsht, sel['5HT2A'])

# For Approach C: φ_D2 and φ_5HT2A from the two heads of the same MLP
# (computed in section 2 as gs_mlp_Cd2 and gs_mlp_Csht)
shap_C_d2_full  = expand_shap_to_fp(gs_mlp_Cd2,  sel['overlap'])
shap_C_sht_full = expand_shap_to_fp(gs_mlp_Csht, sel['overlap'])

# Cosine similarity between receptor-specific vectors per compound
# Approach B SHAP already computed on aligned overlap test set (section 2)
cos_B = cosine_similarity(shap_B_d2_full, shap_B_sht_full).diagonal()
cos_C = cosine_similarity(shap_C_d2_full, shap_C_sht_full).diagonal()

print("EH2 — Target disentanglement")
print(f"  Approach B cos(φ_D2, φ_5HT2A): mean={cos_B.mean():.4f}  std={cos_B.std():.4f}")
print(f"  Approach C cos(φ_D2, φ_5HT2A): mean={cos_C.mean():.4f}  std={cos_C.std():.4f}")
from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(cos_B, cos_C, alternative='two-sided')
print(f"  Mann-Whitney p={p:.4f}  "
      f"({'C more disentangled ✓' if cos_C.mean() < cos_B.mean() else 'B more disentangled ✗'})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cos_B, bins=40, alpha=0.6, color='#D85A30', label=f'B post-hoc  μ={cos_B.mean():.3f}')
ax.hist(cos_C, bins=40, alpha=0.6, color='#534AB7', label=f'C multi-task μ={cos_C.mean():.3f}')
ax.axvline(0, color='black', lw=1, linestyle='--')
ax.set(xlabel='cos(φ_D2, φ_5HT2A)', ylabel='Count',
       title=f'EH2 — Per-receptor attribution overlap  (p={p:.4f})')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_03_eh2_disentanglement.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. EH4 — Attribution entropy (locality)

**Hypothesis:** Direct selectivity models rely on fewer, more localised features.

$$H(\phi_{direct}) < H(\phi_{multi}) < H(\phi_{post})$$

Shannon entropy of the normalised absolute attribution distribution.
Lower entropy = more concentrated = fewer features carry most of the credit.

In [ ]:
def attribution_entropy(shap_mat, active_only=True, smiles_list=None):
    """
    Compute Shannon entropy of normalised |SHAP| per compound.
    If active_only=True, restrict to bits present in the molecule
    (eliminates ghost-bit inflation of entropy).
    """
    entropies = []
    for i in range(len(shap_mat)):
        sv = np.abs(shap_mat[i])

        if active_only and smiles_list is not None:
            mol = Chem.MolFromSmiles(smiles_list[i])
            if mol is not None:
                bi = {}
                rdMolDescriptors.GetMorganFingerprintAsBitVect(
                    mol, 2, nBits=N_FP, bitInfo=bi)
                active = list(bi.keys())
                sv = sv[active]

        total = sv.sum()
        if total == 0:
            continue
        p   = sv / total
        p   = p[p > 0]
        H   = -np.sum(p * np.log2(p))
        entropies.append(H)
    return np.array(entropies)

smiles_te = smiles_te_ov

print("Computing attribution entropy (active bits only)...")
ent = {
    'Tree A (direct)':  attribution_entropy(shap_tree_A, active_only=True, smiles_list=smiles_te),
    'Tree B (post-hoc)':attribution_entropy(shap_tree_B, active_only=True, smiles_list=smiles_te),
    'MLP A (direct)':   attribution_entropy(shap_mlp_A,  active_only=True, smiles_list=smiles_te),
    'MLP B (post-hoc)': attribution_entropy(shap_mlp_B,  active_only=True, smiles_list=smiles_te),
    'MLP C (multi)':    attribution_entropy(shap_mlp_C,  active_only=True, smiles_list=smiles_te),
}

print("\nEH4 — Attribution entropy (lower = more localised)")
print(f"{'Model':22s} | {'Mean H':>8s} | {'Std':>6s}")
print("-"*40)
for name, vals in ent.items():
    print(f"  {name:20s} | {vals.mean():>8.4f} | {vals.std():>6.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, subset, title in [
    (axes[0], ['Tree A (direct)', 'Tree B (post-hoc)'], 'Tree models'),
    (axes[1], ['MLP A (direct)',  'MLP B (post-hoc)', 'MLP C (multi)'], 'MLP models'),
]:
    colors = ['#1D9E75', '#D85A30', '#534AB7']
    for name, col in zip(subset, colors):
        ax.hist(ent[name], bins=30, alpha=0.65, color=col, label=f"{name}  μ={ent[name].mean():.2f}")
    ax.set(xlabel='Shannon entropy H(φ̃)', ylabel='Count', title=f'EH4 — {title}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Attribution entropy — lower = explanations more localised',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_03_eh4_entropy.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. Fragment visualisation

Top global features per approach drawn on example molecules.
Helps verify the explanations identify chemically sensible regions.

In [ ]:
import io
from PIL import Image

def draw_top_bits(smi, shap_vec, top_n=5, size=(600, 400)):
    """
    Highlight top-N active-bit Morgan features.
    top_n kept small so the highlighted region is readable.
    """
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    bi = {}
    rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 2, nBits=N_FP, bitInfo=bi)
    active = set(bi.keys())

    # Rank only active bits so ghost bits don't consume slots
    active_sv  = {b: abs(shap_vec[b]) for b in active}
    top_bits   = sorted(active_sv, key=active_sv.get, reverse=True)[:top_n]

    h_atoms, h_bonds = set(), set()
    a_cols, b_cols   = {}, {}

    for bit in top_bits:
        col = (0.12, 0.72, 0.42) if shap_vec[bit] > 0 else (0.85, 0.30, 0.18)
        for atom_idx, radius in bi[bit]:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
            for bond_idx in env:
                bond = mol.GetBondWithIdx(bond_idx)
                h_bonds.add(bond_idx); b_cols[bond_idx] = col
                h_atoms.add(bond.GetBeginAtomIdx())
                h_atoms.add(bond.GetEndAtomIdx())
            a_cols[atom_idx] = col

    drawer = rdMolDraw2D.MolDraw2DCairo(*size)
    opts   = drawer.drawOptions()
    opts.addAtomIndices   = False
    opts.bondLineWidth     = 2.5
    opts.atomHighlightsAreCircles = False
    try:
        drawer.DrawMolecule(mol, highlightAtoms=list(h_atoms),
                            highlightBonds=list(h_bonds),
                            highlightAtomColors=a_cols,
                            highlightBondColors=b_cols)
        drawer.FinishDrawing()
        return Image.open(io.BytesIO(drawer.GetDrawingText()))
    except Exception:
        return None


# ── Pick 1 atypical + 1 typical from test set ─────────────────────────────────
y_del_te  = merged['delta'].values[te_ov]
idx_atyp  = int(np.argsort(y_del_te)[-1])   # single most atypical
idx_typ   = int(np.argsort(y_del_te)[0])    # single most D2-selective

showcase  = [idx_atyp, idx_typ]
labels    = [f"Most atypical  Δ={y_del_te[idx_atyp]:+.2f}",
             f"Most typical   Δ={y_del_te[idx_typ]:+.2f}"]

approach_data = [
    ('A — direct',   shap_mlp_A),
    ('B — post-hoc', shap_mlp_B),
    ('C — multi',    shap_mlp_C),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Top-5 SHAP features per approach\n'
             'Green = raises ΔpChEMBL (→ atypical)   Red = lowers ΔpChEMBL (→ typical)',
             fontsize=13, fontweight='bold', y=1.01)

for row, (cpd_idx, compound_label) in enumerate(zip(showcase, labels)):
    smi = smiles_te_ov[cpd_idx]
    for col, (appr_label, shap_mat) in enumerate(approach_data):
        ax  = axes[row, col]
        img = draw_top_bits(smi, shap_mat[cpd_idx], top_n=5, size=(700, 450))
        if img:
            ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"Approach {appr_label}\n{compound_label}",
                     fontsize=11, fontweight='bold' if col == 0 else 'normal',
                     pad=8)
        # Subtle border to separate panels
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_color('#D1D5DB')

plt.tight_layout()
plt.savefig('fig_03_fragment_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_03_fragment_maps.png")

In [ ]:
# ── Highlight piperazine linker on the two showcase compounds ─────────────────
from rdkit.Chem import AllChem

PIPERAZINE_SMARTS = Chem.MolFromSmarts('N1CCNCC1')  # piperazine ring
PIPERIDINE_SMARTS = Chem.MolFromSmarts('N1CCCCC1')  # piperidine (single N, typical cpds)

def draw_pharmacophore(smi, size=(700, 450)):
    """Draw molecule with piperazine/piperidine ring highlighted in blue."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None

    h_atoms, h_bonds = set(), set()

    for smarts, col in [
        (PIPERAZINE_SMARTS, (0.18, 0.42, 0.82)),  # blue = piperazine
        (PIPERIDINE_SMARTS, (0.55, 0.18, 0.82)),  # purple = piperidine
    ]:
        matches = mol.GetSubstructMatches(smarts)
        for match in matches:
            for atom_idx in match:
                h_atoms.add(atom_idx)
            for bond in mol.GetBonds():
                if (bond.GetBeginAtomIdx() in match and
                    bond.GetEndAtomIdx()   in match):
                    h_bonds.add(bond.GetIdx())

    a_cols = {a: (0.18, 0.42, 0.82) for a in h_atoms}
    b_cols = {b: (0.18, 0.42, 0.82) for b in h_bonds}

    drawer = rdMolDraw2D.MolDraw2DCairo(*size)
    opts = drawer.drawOptions()
    opts.addAtomIndices = False
    opts.bondLineWidth  = 2.5
    opts.atomHighlightsAreCircles = False
    try:
        drawer.DrawMolecule(mol,
            highlightAtoms=list(h_atoms),
            highlightBonds=list(h_bonds),
            highlightAtomColors=a_cols,
            highlightBondColors=b_cols)
        drawer.FinishDrawing()
        return Image.open(io.BytesIO(drawer.GetDrawingText()))
    except Exception as e:
        print(f"Draw error: {e}")
        return None

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, cpd_idx, label in [
    (axes[0], idx_atyp, f"Most atypical   Δ={y_del_te[idx_atyp]:+.2f}"),
    (axes[1], idx_typ,  f"Most typical    Δ={y_del_te[idx_typ]:+.2f}"),
]:
    smi = smiles_te_ov[cpd_idx]
    img = draw_pharmacophore(smi)
    if img:
        ax.imshow(img)
    ax.axis('off')
    ax.set_title(label, fontsize=12, fontweight='bold', pad=10)

    # Annotate whether piperazine was found
    mol = Chem.MolFromSmiles(smi)
    has_pip = mol.HasSubstructMatch(PIPERAZINE_SMARTS)
    has_pid = mol.HasSubstructMatch(PIPERIDINE_SMARTS)
    ring = "piperazine (N-C-C-N-C-C)" if has_pip else "piperidine (N-C-C-C-C-C)" if has_pid else "no piperazine/piperidine found"
    ax.set_xlabel(f"Blue = {ring}", fontsize=10)

plt.suptitle("Piperazine / piperidine linker highlighted in blue",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_03_piperazine.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Hypothesis summary

In [ ]:
print("="*65)
print("EXPLAINABILITY HYPOTHESES — RESULTS SUMMARY")
print("="*65)

# EH1
print("\nEH1 — Native stability (cross-seed Kendall τ):")
for approach, (tau, std, _) in stability_results.items():
    direction = ('↑' if approach == 'A — direct' else
                 '↓' if approach == 'B — post-hoc' else '~')
    print(f"  {direction} {approach:18s}: τ = {tau:.4f} ± {std:.4f}")
taus_ordered = [stability_results[k][0] for k in
                ['A — direct', 'C — multi', 'B — post-hoc']]
eh1 = "Supported" if taus_ordered[0] >= taus_ordered[1] >= taus_ordered[2] else "Not supported"
print(f"  Verdict: {eh1}")

# EH2
print(f"\nEH2 — Target disentanglement:")
print(f"  B post-hoc cos(φ_D2,φ_5HT2A): {cos_B.mean():.4f}")
print(f"  C multi    cos(φ_D2,φ_5HT2A): {cos_C.mean():.4f}")
eh2 = "Supported" if cos_C.mean() < cos_B.mean() else "Not supported"
print(f"  Verdict: {eh2}  (p={p:.4f})")

# EH4
print(f"\nEH4 — Attribution entropy (locality):")
for name in ['MLP A (direct)', 'MLP C (multi)', 'MLP B (post-hoc)']:
    print(f"  {name:22s}: H = {ent[name].mean():.4f}")
e_A = ent['MLP A (direct)'].mean()
e_C = ent['MLP C (multi)'].mean()
e_B = ent['MLP B (post-hoc)'].mean()
eh4 = "Supported" if e_A <= e_C <= e_B else f"Partial (ordering: A={e_A:.3f} C={e_C:.3f} B={e_B:.3f})"
print(f"  Verdict: {eh4}")

# Save attributions for notebook 04
np.save('shap_tree_A.npy', shap_tree_A)
np.save('shap_tree_B.npy', shap_tree_B)
np.save('shap_mlp_A.npy',  shap_mlp_A)
np.save('shap_mlp_B.npy',  shap_mlp_B)
np.save('shap_mlp_C.npy',  shap_mlp_C)
print("\nSaved: shap_*.npy — load in notebook 04 for benchmark evaluation")